# Kiki SLM — SFT Fine-tuning on Google Colab

**Prerequisites (done locally):**
1. `python scripts/prepare_sft_chatml.py` — download & format 10 SFT datasets
2. Upload `kiki_sft_chatml_train.jsonl` and `kiki_sft_chatml_eval.jsonl` to `My Drive/kiki-slm/data/`
3. Upload `gold_100.jsonl` to `My Drive/kiki-slm/data/` (for evaluation)

**What this notebook does:**
- Calls `scripts/colab_train.py` — trains Qwen3-4B with QLoRA, logs to W&B
- Calls `scripts/colab_eval.py` — compares base vs fine-tuned on gold data
- All processing lives in version-controlled scripts, not in this notebook

## 1. Setup

In [ ]:
%%capture
# Install uv, then all deps via uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os; os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"

!uv pip install --system unsloth wandb datasets
# Pin transformers to 5.2.0 — Unsloth's fast inference is broken on 5.3+
!uv pip install --system transformers==5.2.0

In [ ]:
# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone / update repo and set working directory
import os

REPO_DIR = "/content/drive/MyDrive/kiki-slm/repo"
BRANCH = "feat/loopper-organic-data-pipeline"

if os.path.exists(f"{REPO_DIR}/.git"):
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git reset --hard origin/{BRANCH}
    print(f"Updated repo at {REPO_DIR} (branch: {BRANCH})")
else:
    !git clone -b {BRANCH} https://github.com/BlueSpringsAI/kiki-slm.git {REPO_DIR}
    print(f"Cloned repo to {REPO_DIR} (branch: {BRANCH})")

%cd {REPO_DIR}
!ls scripts/colab_*.py

In [ ]:
# All paths are configured in configs/colab_config.yaml
# Edit THAT file to change paths, hyperparameters, dataset weights, etc.
import yaml, os

with open("configs/colab_config.yaml") as f:
    cfg = yaml.safe_load(f)

TRAIN_FILE = cfg["data"]["train_file"]
EVAL_FILE = cfg["data"]["eval_file"]
GOLD_FILE = cfg["data"]["gold_file"]
DRIVE_OUT = cfg["output"]["adapter_dir"]

# Verify files exist
for label, path in [("Train", TRAIN_FILE), ("Eval", EVAL_FILE)]:
    assert os.path.exists(path), f"{label} not found: {path}"
    print(f"{label}: {os.path.getsize(path)/1024/1024:.1f} MB")

if os.path.exists(GOLD_FILE):
    print(f"Gold:  {sum(1 for l in open(GOLD_FILE) if l.strip())} tickets")
else:
    print(f"Gold:  not found (will use built-in template)")

print(f"\nConfig: configs/colab_config.yaml")
print(f"Model:  {cfg['model']['name']}")
print(f"LoRA:   r={cfg['lora']['r']}, alpha={cfg['lora']['alpha']}")
print(f"Train:  {cfg['training']['epochs']} epochs, lr={cfg['training']['learning_rate']}")

In [ ]:
# Quick data inspection
from datasets import load_dataset
from collections import Counter

ds = load_dataset("json", data_files=TRAIN_FILE, split="train")
print(f"Train: {len(ds):,} examples, columns={ds.column_names}")

if "source" in ds.column_names:
    for src, cnt in sorted(Counter(ds['source']).items(), key=lambda x: -x[1]):
        print(f"  {src:<30s} {cnt:>6,} ({cnt/len(ds)*100:.1f}%)")

del ds  # free memory

## 2. Train

Runs `scripts/colab_train.py` as a subprocess with `python -u` (unbuffered).
- Real-time **tqdm progress bar** + loss logs visible below
- Loss curves, LR schedule, GPU stats tracked in **W&B dashboard**
- Correct step count printed at training start (accounts for packing + grad accum)

In [ ]:
# Train — all config from configs/colab_config.yaml
!python -u scripts/colab_train.py

## 3. Evaluate — Base vs Fine-tuned

Loads base Qwen3 and fine-tuned model **sequentially** (one at a time) to avoid OOM.
Strips `<think>` tokens before JSON parsing.

In [ ]:
# Eval — reads paths from configs/colab_config.yaml
# For checkpoint, find the latest one
import glob
checkpoints = sorted(glob.glob(f"{DRIVE_OUT}/checkpoint-*"))
if checkpoints:
    ADAPTER = checkpoints[-1]
    print(f"Using latest checkpoint: {ADAPTER}")
else:
    ADAPTER = DRIVE_OUT
    print(f"Using adapter dir: {ADAPTER}")

!python -u scripts/colab_eval.py --adapter-path {ADAPTER}